# Prepare YOLO Dataset v2

Подготовка чистого YOLO-датасета из CVAT export `photo_cv`.

Что делает ноутбук:

- читает `data/cvat_exports/photo_cv`;
- проверяет YOLO labels;
- учитывает изображения без рекламы как background;
- делает stratified split `train/val = 80/20`;
- собирает готовый датасет в `data/yolo/ad_surface_v2`;
- пишет `data.yaml` и `split_manifest.csv`.

## 1. Imports and config

In [2]:
from pathlib import Path
import shutil

import pandas as pd
import yaml
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 160)

PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists()),
    Path.cwd(),
)

SOURCE_DATASET_DIR = PROJECT_ROOT / "data" / "cvat_exports" / "photo_cv"
SOURCE_IMAGES_DIR = SOURCE_DATASET_DIR / "images" / "train"
SOURCE_LABELS_DIR = SOURCE_DATASET_DIR / "labels" / "train"

OUTPUT_DATASET_DIR = PROJECT_ROOT / "data" / "yolo" / "ad_surface_v2"

VAL_SIZE = 0.20
RANDOM_STATE = 42
LABEL_ID = 0
LABEL_NAME = "ad_object"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}

DELETE_EXISTING_OUTPUT = True

print("Project root:", PROJECT_ROOT)
print("Source dataset:", SOURCE_DATASET_DIR)
print("Output dataset:", OUTPUT_DATASET_DIR)

Project root: /home/shiawase/ic8_ai/ad_detection
Source dataset: /home/shiawase/ic8_ai/ad_detection/data/cvat_exports/photo_cv
Output dataset: /home/shiawase/ic8_ai/ad_detection/data/yolo/ad_surface_v2


## 2. Check source export

In [2]:
assert SOURCE_DATASET_DIR.exists(), f"Source dataset not found: {SOURCE_DATASET_DIR}"
assert SOURCE_IMAGES_DIR.exists(), f"Images dir not found: {SOURCE_IMAGES_DIR}"
assert SOURCE_LABELS_DIR.exists(), f"Labels dir not found: {SOURCE_LABELS_DIR}"

source_data_yaml_path = SOURCE_DATASET_DIR / "data.yaml"
if source_data_yaml_path.exists():
    print(source_data_yaml_path.read_text(encoding="utf-8"))
else:
    print("Source data.yaml not found, continuing with folder scan")

names:
  0: ad_object
path: .
train: train.txt



## 3. Scan images and validate labels

Изображение без `.txt` label-файла считается background-изображением без рекламы. Для обучения позже будет создан пустой label-файл.

In [3]:
def source_type_for(image_path: Path) -> str:
    return "new_photo" if image_path.name.startswith("photo_") else "old_png"


def box_count_bin(box_count: int) -> str:
    if box_count == 0:
        return "0"
    if box_count == 1:
        return "1"
    if box_count <= 3:
        return "2-3"
    return "4+"


def validate_label_file(label_path: Path) -> list[str]:
    lines = []
    for line_no, line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), start=1):
        line = line.strip()
        if not line:
            continue

        parts = line.split()
        if len(parts) != 5:
            raise ValueError(f"{label_path}:{line_no}: expected 5 columns, got {len(parts)}: {line}")

        class_id = int(parts[0])
        values = [float(value) for value in parts[1:]]

        if class_id != LABEL_ID:
            raise ValueError(f"{label_path}:{line_no}: unexpected class id {class_id}")

        x_center, y_center, width, height = values
        if not all(0.0 <= value <= 1.0 for value in values):
            raise ValueError(f"{label_path}:{line_no}: normalized coordinates out of range: {line}")
        if width <= 0 or height <= 0:
            raise ValueError(f"{label_path}:{line_no}: non-positive bbox size: {line}")

        lines.append(line)

    return lines


image_paths = sorted(
    path for path in SOURCE_IMAGES_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)

rows = []
for image_path in image_paths:
    label_path = SOURCE_LABELS_DIR / f"{image_path.stem}.txt"
    label_exists = label_path.exists()
    label_lines = validate_label_file(label_path) if label_exists else []
    box_count = len(label_lines)

    rows.append({
        "image_name": image_path.name,
        "image_path": str(image_path),
        "label_path": str(label_path) if label_exists else "",
        "label_exists": label_exists,
        "source_type": source_type_for(image_path),
        "box_count": box_count,
        "has_objects": box_count > 0,
        "box_count_bin": box_count_bin(box_count),
    })

metadata_df = pd.DataFrame(rows)

print("Images:", len(metadata_df))
print("Images with objects:", int(metadata_df["has_objects"].sum()))
print("Background images:", int((~metadata_df["has_objects"]).sum()))
metadata_df.head()

Images: 1004
Images with objects: 799
Background images: 205


,image_name,image_path,label_path,label_exists,source_type,box_count,has_objects,box_count_bin
0,001.png,/home/shiawase/ic8_ai/ad_detection/data/cvat_exports/photo_cv/images/train/001.png,/home/shiawase/ic8_ai/ad_detection/data/cvat_exports/photo_cv/labels/train/001.txt,True,old_png,2,True,2-3
1,002.png,/home/shiawase/ic8_ai/ad_detection/data/cvat_exports/photo_cv/images/train/002.png,/home/shiawase/ic8_ai/ad_detection/data/cvat_exports/photo_cv/labels/train/002.txt,True,old_png,4,True,4+
2,003.png,/home/shiawase/ic8_ai/ad_detection/data/cvat_exports/photo_cv/images/train/003.png,/home/shiawase/ic8_ai/ad_detection/data/cvat_exports/photo_cv/labels/train/003.txt,True,old_png,2,True,2-3
3,004.png,/home/shiawase/ic8_ai/ad_detection/data/cvat_exports/photo_cv/images/train/004.png,/home/shiawase/ic8_ai/ad_detection/data/cvat_exports/photo_cv/labels/train/004.txt,True,old_png,5,True,4+
4,005.png,/home/shiawase/ic8_ai/ad_detection/data/cvat_exports/photo_cv/images/train/005.png,/home/shiawase/ic8_ai/ad_detection/data/cvat_exports/photo_cv/labels/train/005.txt,True,old_png,3,True,2-3


## 4. Source distribution

In [4]:
display(pd.crosstab(metadata_df["source_type"], metadata_df["box_count_bin"], margins=True))
display(pd.crosstab(metadata_df["source_type"], metadata_df["has_objects"], margins=True))

metadata_df["box_count"].describe()

box_count_bin,0,1,2-3,4+,All
source_type,,,,,
new_photo,62,120,80,16,278
old_png,143,227,214,142,726
All,205,347,294,158,1004


has_objects,False,True,All
source_type,,,
new_photo,62,216,278
old_png,143,583,726
All,205,799,1004


count    1004.000000
mean        1.855578
std         1.872842
min         0.000000
25%         1.000000
50%         1.000000
75%         3.000000
max        14.000000
Name: box_count, dtype: float64

## 5. Stratified train/val split

Стратифицируем по `source_type + box_count_bin`. Так сохраняется доля старых/новых изображений и background-кадров без рекламы.

In [5]:
metadata_df = metadata_df.copy()
metadata_df["strata"] = metadata_df["source_type"] + "__" + metadata_df["box_count_bin"]

strata_counts = metadata_df["strata"].value_counts().sort_index()
display(strata_counts.to_frame("count"))

if (strata_counts < 2).any():
    raise ValueError("Some strata have fewer than 2 samples; stratified split is not possible")

train_df, val_df = train_test_split(
    metadata_df,
    test_size=VAL_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=metadata_df["strata"],
)

metadata_df.loc[train_df.index, "split"] = "train"
metadata_df.loc[val_df.index, "split"] = "val"
metadata_df = metadata_df.sort_values(["split", "source_type", "image_name"]).reset_index(drop=True)

metadata_df["split"].value_counts()

,count
strata,
new_photo__0,62
new_photo__1,120
new_photo__2-3,80
new_photo__4+,16
old_png__0,143
old_png__1,227
old_png__2-3,214
old_png__4+,142


split
train    803
val      201
Name: count, dtype: int64

## 6. Split quality check

In [6]:
display(pd.crosstab(metadata_df["split"], metadata_df["source_type"], margins=True))
display(pd.crosstab(metadata_df["split"], metadata_df["box_count_bin"], margins=True))
display(pd.crosstab(metadata_df["split"], metadata_df["has_objects"], margins=True))

summary_df = (
    metadata_df
    .groupby(["split", "source_type", "box_count_bin"], dropna=False)
    .size()
    .rename("image_count")
    .reset_index()
)
summary_df

source_type,new_photo,old_png,All
split,,,
train,223,580,803
val,55,146,201
All,278,726,1004


box_count_bin,0,1,2-3,4+,All
split,,,,,
train,164,277,235,127,803
val,41,70,59,31,201
All,205,347,294,158,1004


has_objects,False,True,All
split,,,
train,164,639,803
val,41,160,201
All,205,799,1004


,split,source_type,box_count_bin,image_count
0,train,new_photo,0,50
1,train,new_photo,1,96
2,train,new_photo,2-3,64
3,train,new_photo,4+,13
4,train,old_png,0,114
5,train,old_png,1,181
6,train,old_png,2-3,171
7,train,old_png,4+,114
8,val,new_photo,0,12
9,val,new_photo,1,24


## 7. Build clean YOLO dataset

Исходный CVAT export не меняется. Готовый датасет будет собран в `data/yolo/ad_surface_v2`.

In [7]:
if OUTPUT_DATASET_DIR.exists():
    if DELETE_EXISTING_OUTPUT:
        shutil.rmtree(OUTPUT_DATASET_DIR)
    else:
        raise FileExistsError(f"Output dataset already exists: {OUTPUT_DATASET_DIR}")

for split in ["train", "val"]:
    (OUTPUT_DATASET_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_DATASET_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

for row in metadata_df.itertuples(index=False):
    split = row.split
    src_image_path = Path(row.image_path)
    dst_image_path = OUTPUT_DATASET_DIR / "images" / split / row.image_name
    dst_label_path = OUTPUT_DATASET_DIR / "labels" / split / f"{src_image_path.stem}.txt"

    shutil.copy2(src_image_path, dst_image_path)

    if row.label_exists:
        shutil.copy2(Path(row.label_path), dst_label_path)
    else:
        dst_label_path.write_text("", encoding="utf-8")

print("Dataset written:", OUTPUT_DATASET_DIR)

Dataset written: /home/shiawase/ic8_ai/ad_detection/data/yolo/ad_surface_v2


## 8. Write data.yaml and manifest

In [8]:
data_yaml = {
    "path": str(OUTPUT_DATASET_DIR),
    "train": "images/train",
    "val": "images/val",
    "names": {
        LABEL_ID: LABEL_NAME,
    },
}

data_yaml_path = OUTPUT_DATASET_DIR / "data.yaml"
data_yaml_path.write_text(
    yaml.safe_dump(data_yaml, sort_keys=False, allow_unicode=True),
    encoding="utf-8",
)

manifest_path = OUTPUT_DATASET_DIR / "split_manifest.csv"
metadata_df.to_csv(manifest_path, index=False)

print("data.yaml:", data_yaml_path)
print(data_yaml_path.read_text(encoding="utf-8"))
print("manifest:", manifest_path)

data.yaml: /home/shiawase/ic8_ai/ad_detection/data/yolo/ad_surface_v2/data.yaml
path: /home/shiawase/ic8_ai/ad_detection/data/yolo/ad_surface_v2
train: images/train
val: images/val
names:
  0: ad_object

manifest: /home/shiawase/ic8_ai/ad_detection/data/yolo/ad_surface_v2/split_manifest.csv


## 9. Final verification

In [9]:
def count_files(path: Path, pattern: str = "*") -> int:
    return sum(1 for item in path.glob(pattern) if item.is_file())


checks = []
for split in ["train", "val"]:
    image_dir = OUTPUT_DATASET_DIR / "images" / split
    label_dir = OUTPUT_DATASET_DIR / "labels" / split
    image_count = count_files(image_dir)
    label_count = count_files(label_dir, "*.txt")
    non_empty_label_count = sum(1 for path in label_dir.glob("*.txt") if path.read_text(encoding="utf-8").strip())

    checks.append({
        "split": split,
        "images": image_count,
        "labels": label_count,
        "non_empty_labels": non_empty_label_count,
        "background_labels": label_count - non_empty_label_count,
    })

check_df = pd.DataFrame(checks)
display(check_df)

assert check_df["images"].sum() == len(metadata_df)
assert (check_df["images"] == check_df["labels"]).all(), "Every image must have a label txt, including empty background labels"
assert data_yaml_path.exists()

print("Ready for training:", data_yaml_path)

,split,images,labels,non_empty_labels,background_labels
0,train,803,803,639,164
1,val,201,201,160,41


Ready for training: /home/shiawase/ic8_ai/ad_detection/data/yolo/ad_surface_v2/data.yaml


---

## 10. Train YOLO11x from scratch

Этот блок обучает самую большую YOLO11-архитектуру с нуля: используется `yolo11x.yaml`, а не `.pt`-веса.

Важно: обучение с нуля на 1004 изображениях может дать результат хуже и будет существенно дольше, чем fine-tuning от pretrained weights. Этот блок оставлен именно как эксперимент.

### 10.1 Training config

In [3]:
from ultralytics import YOLO
import torch

TRAIN_DATA_YAML = OUTPUT_DATASET_DIR / "data.yaml"
assert TRAIN_DATA_YAML.exists(), f"data.yaml not found: {TRAIN_DATA_YAML}"

MODEL_CONFIG = "yolo11x.yaml"
TRAIN_PROJECT = PROJECT_ROOT / "runs" / "detect" / "ad_surface_v2"
RUN_NAME = "yolo11x_scratch_img1280"

EPOCHS = 150
IMGSZ = 1280
BATCH = 2
PATIENCE = 30
WORKERS = 4
SEED = 42

DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("Data:", TRAIN_DATA_YAML)
print("Model config:", MODEL_CONFIG)
print("Project:", TRAIN_PROJECT)
print("Run name:", RUN_NAME)
print("Device:", DEVICE)

if DEVICE == "cpu":
    print("WARNING: CUDA is not available. YOLO11x training on CPU will be very slow.")


Data: /home/shiawase/ic8_ai/ad_detection/data/yolo/ad_surface_v2/data.yaml
Model config: yolo11x.yaml
Project: /home/shiawase/ic8_ai/ad_detection/runs/detect/ad_surface_v2
Run name: yolo11x_scratch_img1280
Device: 0


### 10.2 Start training

Эта ячейка запускает долгий процесс обучения.

In [4]:
model = YOLO(MODEL_CONFIG)

train_results = model.train(
    data=str(TRAIN_DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    project=str(TRAIN_PROJECT),
    name=RUN_NAME,
    seed=SEED,
    workers=WORKERS,
    device=DEVICE,
    pretrained=False,
)

train_results


New https://pypi.org/project/ultralytics/8.4.66 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.63 🚀 Python-3.12.3 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 12282MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/shiawase/ic8_ai/ad_detection/data/yolo/ad_surface_v2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x723120bf7680>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

### 10.3 Inspect training artifacts

In [5]:
run_dir = TRAIN_PROJECT / RUN_NAME
results_csv_path = run_dir / "results.csv"
best_weights_path = run_dir / "weights" / "best.pt"
last_weights_path = run_dir / "weights" / "last.pt"

print("Run dir:", run_dir)
print("Results CSV:", results_csv_path, results_csv_path.exists())
print("Best weights:", best_weights_path, best_weights_path.exists())
print("Last weights:", last_weights_path, last_weights_path.exists())

if results_csv_path.exists():
    results_df = pd.read_csv(results_csv_path)
    results_df.columns = results_df.columns.str.strip()
    display(results_df.tail())


Run dir: /home/shiawase/ic8_ai/ad_detection/runs/detect/ad_surface_v2/yolo11x_scratch_img1280
Results CSV: /home/shiawase/ic8_ai/ad_detection/runs/detect/ad_surface_v2/yolo11x_scratch_img1280/results.csv True
Best weights: /home/shiawase/ic8_ai/ad_detection/runs/detect/ad_surface_v2/yolo11x_scratch_img1280/weights/best.pt True
Last weights: /home/shiawase/ic8_ai/ad_detection/runs/detect/ad_surface_v2/yolo11x_scratch_img1280/weights/last.pt True


,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
145,146,19416.4,0.46533,0.39189,0.91097,0.93115,0.89106,0.94394,0.82574,0.52973,0.53492,0.98821,0.000086,0.000086,0.000086
146,147,19531.4,0.44370,0.37399,0.89496,0.93645,0.88268,0.94145,0.82867,0.52987,0.52857,0.98815,0.000073,0.000073,0.000073
147,148,19635.6,0.46491,0.41245,0.90035,0.93508,0.89106,0.94302,0.83245,0.52554,0.52634,0.98569,0.000060,0.000060,0.000060
148,149,19739.8,0.45170,0.37711,0.91745,0.94578,0.87696,0.94485,0.83297,0.52243,0.52173,0.98208,0.000046,0.000046,0.000046
149,150,19855.6,0.44316,0.35985,0.90338,0.93807,0.88846,0.94419,0.83401,0.52018,0.51622,0.98036,0.000033,0.000033,0.000033


### 10.4 Plot metrics

In [6]:
import plotly.express as px

if results_csv_path.exists():
    results_df = pd.read_csv(results_csv_path)
    results_df.columns = results_df.columns.str.strip()

    metric_columns = [
        "metrics/precision(B)",
        "metrics/recall(B)",
        "metrics/mAP50(B)",
        "metrics/mAP50-95(B)",
    ]
    available_metric_columns = [column for column in metric_columns if column in results_df.columns]

    if available_metric_columns:
        metrics_long_df = results_df[["epoch", *available_metric_columns]].melt(
            id_vars="epoch",
            var_name="metric",
            value_name="value",
        )
        fig = px.line(metrics_long_df, x="epoch", y="value", color="metric")
        fig.show()
    else:
        print("No metric columns found in results.csv")
else:
    print("Run training first")


### 10.5 Plot losses

In [8]:
if results_csv_path.exists():
    results_df = pd.read_csv(results_csv_path)
    results_df.columns = results_df.columns.str.strip()

    loss_columns = [
        "train/box_loss",
        "train/cls_loss",
        "train/dfl_loss",
        "val/box_loss",
        "val/cls_loss",
        "val/dfl_loss",
    ]
    available_loss_columns = [column for column in loss_columns if column in results_df.columns]

    if available_loss_columns:
        loss_long_df = results_df[["epoch", *available_loss_columns]].melt(
            id_vars="epoch",
            var_name="loss",
            value_name="value",
        )
        fig = px.line(loss_long_df, x="epoch", y="value", color="loss")
        fig.show()
    else:
        print("No loss columns found in results.csv")
else:
    print("Run training first")


### 10.6 Validate best weights

In [7]:
if best_weights_path.exists():
    best_model = YOLO(str(best_weights_path))
    val_results = best_model.val(
        data=str(TRAIN_DATA_YAML),
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        split="val",
    )
    val_results
else:
    print("Best weights not found. Run training first.")


Ultralytics 8.4.63 🚀 Python-3.12.3 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 12282MiB)
YOLO11x summary (fused): 191 layers, 56,828,179 parameters, 0 gradients, 194.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 7114.8±2268.4 MB/s, size: 3874.6 KB)
val: Scanning /home/shiawase/ic8_ai/ad_detection/data/yolo/ad_surface_v2/labels/val.cache... 201 images, 41 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 201/201 84.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 101/101 14.0it/s 7.2s.1s
                   all        201        358      0.938       0.89      0.944      0.834
Speed: 1.7ms preprocess, 29.6ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to /home/shiawase/ic8_ai/ad_detection/runs/detect/val


---

## 11. Video inference

Прогоняем новое видео через обученную модель `yolo11x_scratch_img1280`. Результаты сохраняются в `runs/detect/ad_surface_video/`: аннотированное видео и CSV с bbox по кадрам.

### 11.1 Inference config

In [ ]:
from pathlib import Path

import cv2
import pandas as pd
import torch
from IPython.display import Video, display
from ultralytics import YOLO

VIDEO_PATH = PROJECT_ROOT / "test.mp4"
VIDEO_MODEL_PATH = PROJECT_ROOT / "models" / "trained" / "yolo11x_scratch_img1280" / "best.pt"
VIDEO_PROJECT = PROJECT_ROOT / "runs" / "detect" / "ad_surface_video"
VIDEO_RUN_NAME = "test_yolo11x_detect25_show55"

# Detection threshold: keep more candidates for analysis.
VIDEO_CONF = 0.25

# Display threshold: draw only confident boxes on the output video.
VIDEO_DISPLAY_CONF = 0.55

VIDEO_IMGSZ = 1280
VIDEO_VID_STRIDE = 1
VIDEO_DEVICE = 0 if torch.cuda.is_available() else "cpu"

assert VIDEO_PATH.exists(), f"Video not found: {VIDEO_PATH}"
assert VIDEO_MODEL_PATH.exists(), f"Model weights not found: {VIDEO_MODEL_PATH}"

print("Video:", VIDEO_PATH)
print("Model:", VIDEO_MODEL_PATH)
print("Output project:", VIDEO_PROJECT)
print("Run name:", VIDEO_RUN_NAME)
print("Detection conf:", VIDEO_CONF)
print("Display conf:", VIDEO_DISPLAY_CONF)
print("Device:", VIDEO_DEVICE)


### 11.2 Video metadata

In [12]:
capture = cv2.VideoCapture(str(VIDEO_PATH))
assert capture.isOpened(), f"Could not open video: {VIDEO_PATH}"

fps = capture.get(cv2.CAP_PROP_FPS)
frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration_sec = frame_count / fps if fps else None
capture.release()

video_info = {
    "path": str(VIDEO_PATH),
    "fps": fps,
    "frame_count": frame_count,
    "duration_sec": duration_sec,
    "width": width,
    "height": height,
}

pd.DataFrame([video_info])


,path,fps,frame_count,duration_sec,width,height
0,/home/shiawase/ic8_ai/ad_detection/test.mp4,29.992732,11182,372.823656,720,1280


### 11.3 Run prediction and save annotated video

Эта ячейка может выполняться долго. Она проходит по видео потоково, поэтому не держит все кадры в памяти.

In [ ]:
video_model = YOLO(str(VIDEO_MODEL_PATH))
video_save_dir = VIDEO_PROJECT / VIDEO_RUN_NAME
video_save_dir.mkdir(parents=True, exist_ok=True)

video_detections_csv = VIDEO_PROJECT / f"{VIDEO_RUN_NAME}_detections.csv"
video_output_path = video_save_dir / f"{VIDEO_PATH.stem}_{VIDEO_RUN_NAME}.mp4"

output_fps = fps if fps and fps > 0 else 25
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(video_output_path), fourcc, output_fps, (width, height))
assert writer.isOpened(), f"Could not create output video: {video_output_path}"

prediction_stream = video_model.predict(
    source=str(VIDEO_PATH),
    conf=VIDEO_CONF,
    imgsz=VIDEO_IMGSZ,
    vid_stride=VIDEO_VID_STRIDE,
    save=False,
    save_txt=False,
    save_conf=False,
    device=VIDEO_DEVICE,
    stream=True,
    verbose=False,
)

rows = []
for frame_index, result in enumerate(prediction_stream):
    if frame_index == 0 or (frame_index + 1) % 250 == 0:
        print(f"Processed frame {frame_index + 1}/{frame_count}")

    frame = result.orig_img.copy()
    if frame.shape[1] != width or frame.shape[0] != height:
        frame = cv2.resize(frame, (width, height))

    if result.boxes is not None and len(result.boxes) > 0:
        boxes = result.boxes.xyxy.cpu().tolist()
        confidences = result.boxes.conf.cpu().tolist()
        class_ids = result.boxes.cls.cpu().tolist()

        for box, confidence, class_id_float in zip(boxes, confidences, class_ids, strict=True):
            class_id = int(class_id_float)
            label = result.names.get(class_id, str(class_id))
            x1, y1, x2, y2 = [float(value) for value in box]
            displayed = confidence >= VIDEO_DISPLAY_CONF

            rows.append({
                "frame": frame_index,
                "time_sec": frame_index / fps if fps else None,
                "class_id": class_id,
                "label": label,
                "confidence": float(confidence),
                "displayed": displayed,
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "box_width": x2 - x1,
                "box_height": y2 - y1,
            })

            if displayed:
                p1 = (int(round(x1)), int(round(y1)))
                p2 = (int(round(x2)), int(round(y2)))
                color = (0, 0, 255)
                cv2.rectangle(frame, p1, p2, color, 2)

                caption = f"{label} {confidence:.2f}"
                text_origin = (p1[0], max(p1[1] - 8, 20))
                cv2.putText(
                    frame,
                    caption,
                    text_origin,
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    color,
                    2,
                    cv2.LINE_AA,
                )

    writer.write(frame)

writer.release()

detections_df = pd.DataFrame(rows)
video_detections_csv.parent.mkdir(parents=True, exist_ok=True)
detections_df.to_csv(video_detections_csv, index=False)

print("Save dir:", video_save_dir)
print("Output video:", video_output_path)
print("Detections CSV:", video_detections_csv)
print("Detections >= detection conf:", len(detections_df))
print("Detections displayed:", int(detections_df["displayed"].sum()) if not detections_df.empty else 0)
detections_df.head()


### 11.4 Find and preview output video

In [14]:
video_outputs = sorted(
    [
        path for path in video_save_dir.rglob("*")
        if path.suffix.lower() in {".mp4", ".avi", ".mov", ".mkv", ".webm"}
    ],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

print("Output videos:")
for path in video_outputs:
    print(path)

if video_outputs:
    display(Video(str(video_outputs[0]), embed=False, width=960))
else:
    print("No output video found. Check the previous cell output and save_dir.")


Output videos:
/home/shiawase/ic8_ai/ad_detection/runs/detect/ad_surface_video/test_yolo11x_conf60/test.avi


### 11.5 Detection summary

In [ ]:
if video_detections_csv.exists():
    detections_df = pd.read_csv(video_detections_csv)
    display(detections_df.describe(include="all"))

    if not detections_df.empty:
        display(detections_df["displayed"].value_counts().rename_axis("displayed").reset_index(name="detections"))

        per_frame_df = (
            detections_df
            .groupby("frame")
            .agg(
                detections=("frame", "size"),
                displayed_detections=("displayed", "sum"),
            )
            .reset_index()
        )
        display(per_frame_df.describe())
else:
    print("Run prediction first")


### 11.6 Optional: tracking with ByteTrack

Если нужны стабильные ID объектов между кадрами, можно запустить tracking. Эта ячейка сохраняет только CSV с треками, без автоотрисовки видео, чтобы не показывать bbox ниже `VIDEO_DISPLAY_CONF`.

In [ ]:
TRACK_RUN_NAME = "test_yolo11x_track_conf25"
TRACKER = "bytetrack.yaml"
TRACK_PROJECT = PROJECT_ROOT / "runs" / "detect" / "ad_surface_video"
TRACK_CSV = TRACK_PROJECT / f"{TRACK_RUN_NAME}_tracks.csv"

track_stream = video_model.track(
    source=str(VIDEO_PATH),
    conf=VIDEO_CONF,
    imgsz=VIDEO_IMGSZ,
    vid_stride=VIDEO_VID_STRIDE,
    tracker=TRACKER,
    persist=True,
    save=False,
    save_txt=False,
    save_conf=False,
    project=str(TRACK_PROJECT),
    name=TRACK_RUN_NAME,
    exist_ok=True,
    device=VIDEO_DEVICE,
    stream=True,
    verbose=False,
)

track_rows = []
for frame_index, result in enumerate(track_stream):
    if frame_index == 0 or (frame_index + 1) % 250 == 0:
        print(f"Tracked frame {frame_index + 1}/{frame_count}")

    if result.boxes is None or len(result.boxes) == 0:
        continue

    boxes = result.boxes.xyxy.cpu().tolist()
    confidences = result.boxes.conf.cpu().tolist()
    class_ids = result.boxes.cls.cpu().tolist()
    track_ids = result.boxes.id.cpu().tolist() if result.boxes.id is not None else [None] * len(boxes)

    for box, confidence, class_id_float, track_id in zip(boxes, confidences, class_ids, track_ids, strict=True):
        class_id = int(class_id_float)
        label = result.names.get(class_id, str(class_id))
        x1, y1, x2, y2 = [float(value) for value in box]
        displayed = confidence >= VIDEO_DISPLAY_CONF

        track_rows.append({
            "frame": frame_index,
            "time_sec": frame_index / fps if fps else None,
            "track_id": int(track_id) if track_id is not None else None,
            "class_id": class_id,
            "label": label,
            "confidence": float(confidence),
            "displayed": displayed,
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
            "box_width": x2 - x1,
            "box_height": y2 - y1,
        })

tracks_df = pd.DataFrame(track_rows)
TRACK_CSV.parent.mkdir(parents=True, exist_ok=True)
tracks_df.to_csv(TRACK_CSV, index=False)

print("Track CSV:", TRACK_CSV)
print("Tracks rows:", len(tracks_df))
tracks_df.head()


## 12. Export compact predicted video frames for CVAT

Собираем компактный набор кадров из видео для ручной проверки в CVAT.

Идея: не сохранять все 3700 похожих кадров подряд, а брать только разреженные и визуально отличающиеся кадры, где модель нашла bbox. Такой набор удобен для hard negatives: в CVAT удаляем ложные bbox на стенах/салоне/фасадах, оставляем или правим настоящие рекламные поверхности.

### 12.1 Compact CVAT export config

In [ ]:
import shutil
import xml.etree.ElementTree as ET
from datetime import UTC, datetime
from zipfile import ZIP_STORED, ZipFile

CVAT_VIDEO_EXPORT_DIR = PROJECT_ROOT / "cvat_import" / "video_predict_compact"
CVAT_VIDEO_IMAGES_DIR = CVAT_VIDEO_EXPORT_DIR / "images"
CVAT_VIDEO_XML_PATH = CVAT_VIDEO_EXPORT_DIR / "annotations.xml"
CVAT_VIDEO_ZIP_PATH = CVAT_VIDEO_EXPORT_DIR / "images.zip"
CVAT_VIDEO_MANIFEST_PATH = CVAT_VIDEO_EXPORT_DIR / "frame_manifest.csv"

# Keep prediction candidates from this threshold in CVAT XML.
CVAT_EXPORT_CONF = 0.25

# Save frames only when they have at least one bbox.
CVAT_ONLY_FRAMES_WITH_BOXES = True

# Optional: also save every N-th frame without bbox.
# Set to None to disable. Useful for missed billboards that model did not detect.
CVAT_BACKGROUND_SAMPLE_STRIDE = None

# Compact sampling controls.
CVAT_SKIP_START_SECONDS = 0.0      # set e.g. 20.0 to skip a stationary beginning
CVAT_MIN_FRAME_GAP = 60           # about 2 seconds for a 30 fps video
CVAT_MIN_VISUAL_DIFF = 0.08       # compare against recent saved frames, 0..1 grayscale diff
CVAT_RECENT_SIGNATURE_COUNT = 8   # compare with several previous saved frames, not only the last one
CVAT_TIME_BUCKET_SECONDS = 10.0   # temporal bucket for quota limiting
CVAT_MAX_FRAMES_PER_BUCKET = 2    # prevents one stationary segment from dominating the task
CVAT_MAX_FRAMES = 250             # target cap for manual CVAT review

CVAT_FRAME_PREFIX = "video"
CVAT_LABEL_NAME = "ad_object"
CVAT_CLEAN_OUTPUT = True

print("CVAT compact export dir:", CVAT_VIDEO_EXPORT_DIR)
print("XML:", CVAT_VIDEO_XML_PATH)
print("Images zip:", CVAT_VIDEO_ZIP_PATH)
print("Prediction conf:", CVAT_EXPORT_CONF)
print("Skip start seconds:", CVAT_SKIP_START_SECONDS)
print("Min frame gap:", CVAT_MIN_FRAME_GAP)
print("Min visual diff:", CVAT_MIN_VISUAL_DIFF)
print("Max frames per bucket:", CVAT_MAX_FRAMES_PER_BUCKET, "per", CVAT_TIME_BUCKET_SECONDS, "sec")
print("Max frames:", CVAT_MAX_FRAMES)


### 12.2 Helper functions for CVAT XML and deduplication

In [ ]:
def append_text(parent: ET.Element, tag: str, text: str | int) -> ET.Element:
    child = ET.SubElement(parent, tag)
    child.text = str(text)
    return child


def frame_signature(frame, size: tuple[int, int] = (64, 64)):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    small = cv2.resize(gray, size, interpolation=cv2.INTER_AREA)
    return small


def frame_difference_score(signature_a, signature_b) -> float:
    if signature_a is None or signature_b is None:
        return 1.0
    return float(cv2.absdiff(signature_a, signature_b).mean() / 255.0)


def min_recent_frame_difference(signature, recent_signatures: list) -> float:
    if not recent_signatures:
        return 1.0
    return min(frame_difference_score(signature, recent_signature) for recent_signature in recent_signatures)


def build_cvat_xml_from_video_frames(
    frame_rows: list[dict],
    detections_by_name: dict[str, list[dict]],
    label_name: str,
) -> ET.ElementTree:
    now = datetime.now(UTC).isoformat()
    annotations = ET.Element("annotations")
    append_text(annotations, "version", "1.1")

    meta = ET.SubElement(annotations, "meta")
    task = ET.SubElement(meta, "task")
    append_text(task, "id", 0)
    append_text(task, "name", "video_predict_compact_frames")
    append_text(task, "size", len(frame_rows))
    append_text(task, "mode", "annotation")
    append_text(task, "overlap", 0)
    append_text(task, "bugtracker", "")
    append_text(task, "created", now)
    append_text(task, "updated", now)
    append_text(task, "subset", "default")
    append_text(task, "start_frame", 0)
    append_text(task, "stop_frame", max(len(frame_rows) - 1, 0))
    append_text(task, "frame_filter", "")

    segments = ET.SubElement(task, "segments")
    segment = ET.SubElement(segments, "segment")
    append_text(segment, "id", 0)
    append_text(segment, "start", 0)
    append_text(segment, "stop", max(len(frame_rows) - 1, 0))
    append_text(segment, "url", "")

    owner = ET.SubElement(task, "owner")
    append_text(owner, "username", "")
    append_text(owner, "email", "")
    append_text(task, "assignee", "")

    labels_node = ET.SubElement(task, "labels")
    label = ET.SubElement(labels_node, "label")
    append_text(label, "name", label_name)
    append_text(label, "color", "#fa3253")
    append_text(label, "type", "rectangle")
    ET.SubElement(label, "attributes")

    append_text(meta, "dumped", now)

    for image_id, frame_row in enumerate(frame_rows):
        image_name = frame_row["image_name"]
        image_node = ET.SubElement(
            annotations,
            "image",
            {
                "id": str(image_id),
                "name": image_name,
                "width": str(frame_row["width"]),
                "height": str(frame_row["height"]),
            },
        )

        for detection in detections_by_name.get(image_name, []):
            ET.SubElement(
                image_node,
                "box",
                {
                    "label": label_name,
                    "source": "auto",
                    "occluded": "0",
                    "xtl": f"{detection['x1']:.2f}",
                    "ytl": f"{detection['y1']:.2f}",
                    "xbr": f"{detection['x2']:.2f}",
                    "ybr": f"{detection['y2']:.2f}",
                    "z_order": "0",
                },
            )

    ET.indent(annotations, space="  ")
    return ET.ElementTree(annotations)


### 12.3 Run predict and save compact CVAT frames

Эта ячейка сохраняет компактный набор кадров и XML. Для CVAT создай task из `images.zip`, затем импортируй `annotations.xml` в формате CVAT for images.

In [ ]:
if CVAT_CLEAN_OUTPUT and CVAT_VIDEO_EXPORT_DIR.exists():
    shutil.rmtree(CVAT_VIDEO_EXPORT_DIR)
CVAT_VIDEO_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

assert VIDEO_PATH.exists(), f"Video not found: {VIDEO_PATH}"
assert VIDEO_MODEL_PATH.exists(), f"Model weights not found: {VIDEO_MODEL_PATH}"

cvat_model = YOLO(str(VIDEO_MODEL_PATH))

prediction_stream = cvat_model.predict(
    source=str(VIDEO_PATH),
    conf=CVAT_EXPORT_CONF,
    imgsz=VIDEO_IMGSZ,
    vid_stride=VIDEO_VID_STRIDE,
    save=False,
    save_txt=False,
    save_conf=False,
    device=VIDEO_DEVICE,
    stream=True,
    verbose=False,
)

frame_rows = []
detections_by_name = {}
all_detection_rows = []
last_saved_frame_index = None
recent_signatures = []
time_bucket_counts = {}
skip_reasons = {
    "before_start": 0,
    "no_candidate": 0,
    "time_bucket_quota": 0,
    "frame_gap": 0,
    "visual_duplicate": 0,
    "max_frames": 0,
}

for frame_index, result in enumerate(prediction_stream):
    if CVAT_MAX_FRAMES is not None and len(frame_rows) >= CVAT_MAX_FRAMES:
        skip_reasons["max_frames"] += 1
        break

    source_time_sec = frame_index / fps if fps else None
    if source_time_sec is not None and source_time_sec < CVAT_SKIP_START_SECONDS:
        skip_reasons["before_start"] += 1
        continue

    if frame_index == 0 or (frame_index + 1) % 250 == 0:
        print(f"Scanned frame {frame_index + 1}/{frame_count}; saved {len(frame_rows)}")

    frame = result.orig_img.copy()
    frame_height, frame_width = frame.shape[:2]

    frame_detections = []
    if result.boxes is not None and len(result.boxes) > 0:
        boxes = result.boxes.xyxy.cpu().tolist()
        confidences = result.boxes.conf.cpu().tolist()
        class_ids = result.boxes.cls.cpu().tolist()

        for box, confidence, class_id_float in zip(boxes, confidences, class_ids, strict=True):
            class_id = int(class_id_float)
            label = result.names.get(class_id, str(class_id))
            x1, y1, x2, y2 = [float(value) for value in box]

            detection = {
                "frame": frame_index,
                "time_sec": source_time_sec,
                "class_id": class_id,
                "label": label,
                "confidence": float(confidence),
                "x1": max(0.0, min(x1, float(frame_width))),
                "y1": max(0.0, min(y1, float(frame_height))),
                "x2": max(0.0, min(x2, float(frame_width))),
                "y2": max(0.0, min(y2, float(frame_height))),
            }
            if detection["x2"] > detection["x1"] and detection["y2"] > detection["y1"]:
                frame_detections.append(detection)
                all_detection_rows.append(detection)

    has_boxes = len(frame_detections) > 0
    sample_background = (
        CVAT_BACKGROUND_SAMPLE_STRIDE is not None
        and CVAT_BACKGROUND_SAMPLE_STRIDE > 0
        and frame_index % CVAT_BACKGROUND_SAMPLE_STRIDE == 0
    )
    is_candidate = has_boxes or sample_background or (not CVAT_ONLY_FRAMES_WITH_BOXES and sample_background)

    if not is_candidate:
        skip_reasons["no_candidate"] += 1
        continue

    if source_time_sec is not None and CVAT_TIME_BUCKET_SECONDS and CVAT_MAX_FRAMES_PER_BUCKET is not None:
        time_bucket = int(source_time_sec // CVAT_TIME_BUCKET_SECONDS)
        if time_bucket_counts.get(time_bucket, 0) >= CVAT_MAX_FRAMES_PER_BUCKET:
            skip_reasons["time_bucket_quota"] += 1
            continue
    else:
        time_bucket = None

    if last_saved_frame_index is not None and CVAT_MIN_FRAME_GAP is not None:
        if frame_index - last_saved_frame_index < CVAT_MIN_FRAME_GAP:
            skip_reasons["frame_gap"] += 1
            continue

    signature = frame_signature(frame)
    visual_diff = min_recent_frame_difference(signature, recent_signatures)
    if recent_signatures and CVAT_MIN_VISUAL_DIFF is not None:
        if visual_diff < CVAT_MIN_VISUAL_DIFF:
            skip_reasons["visual_duplicate"] += 1
            continue

    image_name = f"{CVAT_FRAME_PREFIX}_{len(frame_rows) + 1:06d}.jpg"
    image_path = CVAT_VIDEO_IMAGES_DIR / image_name
    cv2.imwrite(str(image_path), frame)

    frame_rows.append({
        "image_name": image_name,
        "source_frame": frame_index,
        "time_sec": source_time_sec,
        "width": frame_width,
        "height": frame_height,
        "box_count": len(frame_detections),
        "visual_diff_from_recent_saved": visual_diff,
        "time_bucket": time_bucket,
    })
    detections_by_name[image_name] = frame_detections
    last_saved_frame_index = frame_index
    recent_signatures.append(signature)
    recent_signatures = recent_signatures[-CVAT_RECENT_SIGNATURE_COUNT:]
    if time_bucket is not None:
        time_bucket_counts[time_bucket] = time_bucket_counts.get(time_bucket, 0) + 1

xml_tree = build_cvat_xml_from_video_frames(
    frame_rows=frame_rows,
    detections_by_name=detections_by_name,
    label_name=CVAT_LABEL_NAME,
)
xml_tree.write(CVAT_VIDEO_XML_PATH, encoding="utf-8", xml_declaration=True)

with ZipFile(CVAT_VIDEO_ZIP_PATH, "w", compression=ZIP_STORED) as zip_file:
    for frame_row in frame_rows:
        image_path = CVAT_VIDEO_IMAGES_DIR / frame_row["image_name"]
        zip_file.write(image_path, arcname=frame_row["image_name"])

frame_manifest_df = pd.DataFrame(frame_rows)
frame_manifest_df.to_csv(CVAT_VIDEO_MANIFEST_PATH, index=False)

all_detections_df = pd.DataFrame(all_detection_rows)
all_detections_csv = CVAT_VIDEO_EXPORT_DIR / "all_detections.csv"
all_detections_df.to_csv(all_detections_csv, index=False)

print("Saved frames:", len(frame_rows))
print("Frames with boxes:", int((frame_manifest_df["box_count"] > 0).sum()) if not frame_manifest_df.empty else 0)
print("Boxes in XML:", sum(len(items) for items in detections_by_name.values()))
print("Skip reasons:", skip_reasons)
print("Images zip:", CVAT_VIDEO_ZIP_PATH)
print("Annotations XML:", CVAT_VIDEO_XML_PATH)
print("Frame manifest:", CVAT_VIDEO_MANIFEST_PATH)
print("All detections CSV:", all_detections_csv)

frame_manifest_df.head()


### 12.4 Inspect compact CVAT files

In [ ]:
exported_images = sorted(CVAT_VIDEO_IMAGES_DIR.glob("*.jpg"))
print("Export dir:", CVAT_VIDEO_EXPORT_DIR)
print("Image files:", len(exported_images))
print("Zip exists:", CVAT_VIDEO_ZIP_PATH.exists(), CVAT_VIDEO_ZIP_PATH)
print("XML exists:", CVAT_VIDEO_XML_PATH.exists(), CVAT_VIDEO_XML_PATH)

if exported_images:
    print("First images:", [path.name for path in exported_images[:10]])

if CVAT_VIDEO_MANIFEST_PATH.exists():
    manifest_df = pd.read_csv(CVAT_VIDEO_MANIFEST_PATH)
    display(manifest_df["box_count"].value_counts().sort_index().rename_axis("box_count").reset_index(name="frames"))
    display(manifest_df.describe())
    display(manifest_df.head())
